# <p style="text-align: center; font-size: 2.5rem">Notebook I</p>

<p style="text-align: center; font-size: 2rem">Part a</p>

This notebook contains the cleaning and wrangling of the Marine Mammal Strandings Dataset for animals stranded along the south-east coast of the United States.

Data should be ingested from the `data/raw/UNC-DataRequest-01302026.xlsx` file and any files containing cleaned data should go in the `data/processed/` directory.


## Notebook Setup


Library imports mainly focusing on data viewing and cleaning along with other helpful modules.


In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path

Constants mainly pertaining to directory paths.


In [2]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

Helpful functions for repeated insights


In [3]:
def print_shape(shape: tuple[int, ...]) -> None:
    print(f"The dataset has {shape[0]:,} rows and {shape[1]:,} columns")


def print_time_frame(col: pd.Series) -> None:
    min_year = col.min()
    max_year = col.max()
    print(f"The dataset has entries from {min_year} to {max_year}")

## Loading Datasets


While we have a primary dataset with marine mammal strandings along the southeast coast, the example dataset of Bottlenose Dolphins from 2017 to 2019 was also retrieved for a quick comparison of dataset traits.


Loading the example dataset and keeping records where the state is one of our target states. Then viewing the dataset's shape.


In [4]:
example_df = pd.read_excel(
    RAW_DATA_DIR / "EastCoast-Bottlenose-2017-2019.xlsx", sheet_name="Dataset"
)
example_df = example_df[example_df["State"].str.lower().isin(["nc", "sc", "va"])]

In [5]:
print(example_df.columns.sort_values())

Index(['2007 Partial Carcass', '2007 Whole Carcass', 'Abandoned/Orphaned Flag',
       'Additional Remarks', 'Age Class', 'Common Name', 'Condition Comments',
       'Condition at Examination', 'Confidence Code', 'County', 'Date of Exam',
       'Date of Examination', 'Day of Observation', 'Dead Animal Exam',
       'Deemed Healthy/Releasable Flag', 'Died During Transport Flag',
       'Died at Site Flag', 'Euthanized During Transport',
       'Euthanized at Site Flag', 'Examined', 'Extent of Exam', 'Genus',
       'How lat/long determined', 'Immediate Release at Site',
       'Inaccesible Flag', 'Injured Flag', 'Latitude',
       'Latitude Actual/Estimate', 'Latitude Units', 'Left at Site', 'Length',
       'Length Units', 'Length actual/estimate', 'Location Hazardous Public',
       'Location Hazardous to Animal', 'Longitude',
       'Longitude Actual/Estimate', 'Longitude Units', 'Month of Observation',
       'National Database Number', 'Necropy Carcass Fresh/Frozen',
       'Obser

In [6]:
print_shape(example_df.shape)

The dataset has 2,550 rows and 55 columns


In [7]:
print_time_frame(example_df["Year of Observation"])

The dataset has entries from 2015 to 2024


Loading the main dataset and viewing its shape.


In [8]:
df = pd.read_excel(
    RAW_DATA_DIR / "UNC-DataRequest-01302026.xlsx", sheet_name="2015-2024"
)

In [9]:
print(df.columns.sort_values())

Index(['2007 Partial Carcass', '2007 Whole Carcass', 'Abandoned/Orphaned Flag',
       'Additional Remarks', 'Age Class', 'Common Name', 'Condition Comments',
       'Condition at Examination', 'Confidence Code', 'County', 'Date of Exam',
       'Date of Examination', 'Day of Observation', 'Dead Animal Exam',
       'Deemed Healthy/Releasable Flag', 'Died During Transport Flag',
       'Died at Site Flag', 'Euthanized During Transport',
       'Euthanized at Site Flag', 'Examined', 'Extent of Exam', 'Genus',
       'How lat/long determined', 'Immediate Release at Site',
       'Inaccesible Flag', 'Injured Flag', 'Latitude',
       'Latitude Actual/Estimate', 'Latitude Units', 'Left at Site', 'Length',
       'Length Units', 'Length actual/estimate', 'Location Hazardous Public',
       'Location Hazardous to Animal', 'Longitude',
       'Longitude Actual/Estimate', 'Longitude Units', 'Month of Observation',
       'National Database Number', 'Necropy Carcass Fresh/Frozen',
       'Obser

In [10]:
print_shape(df.shape)

The dataset has 2,550 rows and 55 columns


In [11]:
print_time_frame(df["Year of Observation"])

The dataset has entries from 2015 to 2024


As expected, the actual dataset had far more data about strandings as it looks at strandings from a larger time frame (~9 years) compared the example dataset (~2 years).

Unexpectedly, the number of columns in the datasets differ, being that the example dataset (69 columns) has more columns than the actual dataset (55 columns).

This difference is investigated to determine if this may result in a loss of major data.


## Column Comparison


To see the difference in columns, the columns of the dataframes are turned into sets and simple set operations are performed.


In [12]:
example_columns = set(example_df.columns)
actual_columns = set(df.columns)

The formula below, with $A$ being the columns from one dataset and $B$ being columns from the other dataset, will produce the columns that are only in the first dataset.

$$
A \setminus B = \{x \mid x \in A \text{ and } x \notin B\}
$$


In [13]:
actual_only = actual_columns - example_columns
print(f"Columns only in actual data ({len(actual_only)} columns):")

for col in sorted(actual_only):
    print(f"- {col}")

Columns only in actual data (0 columns):


In [14]:
example_only = example_columns - actual_columns
print(f"Columns only in example data ({len(example_only)} columns):")

for col in sorted(example_only):
    print(f"- {col}")

Columns only in example data (0 columns):


There is a significant difference between the columns of the datasets. Most of this is minor differences in column names that refer to the same idea and other variables like about the examination of the mammal but features about the observations and the condition of the animal of observation are missing such as _Boat Collision_ and _How Observed_.

If these features could provide significant influence to the predictions of strandings, then the data will be requested again as the origin of the issue is likely due to the query made to the NOAA.


## Filtering the Datasets


In [15]:
df.info(show_counts=False)

<class 'pandas.DataFrame'>
RangeIndex: 2550 entries, 0 to 2549
Data columns (total 55 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   National Database Number        str    
 1   Confidence Code                 str    
 2   Common Name                     str    
 3   Genus                           str    
 4   Species                         str    
 5   County                          str    
 6   State                           str    
 7   Latitude                        float64
 8   Latitude Units                  str    
 9   Latitude Actual/Estimate        str    
 10  Longitude                       object 
 11  Longitude Units                 str    
 12  Longitude Actual/Estimate       str    
 13  How lat/long determined         str    
 14  Year of Observation             int64  
 15  Month of Observation            str    
 16  Day of Observation              int64  
 17  Observation Status              str    
 18 

### Checking Longitude and Latitude Coordinates


Only latitude and longitude coordinates with the same measurement accuracy type are kept in order to reduce complications.


In [16]:
df["Longitude Actual/Estimate"].value_counts()

Longitude Actual/Estimate
actual      2000
estimate     531
Name: count, dtype: int64

In [17]:
df["Latitude Actual/Estimate"].value_counts()

Latitude Actual/Estimate
actual      2011
estimate     530
Name: count, dtype: int64

In [18]:
df = df[df["Longitude Actual/Estimate"] == df["Latitude Actual/Estimate"]].reset_index(
    drop=True
)

A flag for if coordinate pairs are actual or an estimate is made for simplicity.


In [19]:
df["coords_is_actual"] = df["Latitude Actual/Estimate"] == "actual"

Referring to the information about each column and their data types, the latitudes were all floats but some records in the longitudes caused it to be designated as a columns of objects.

To determine which records had non-float longitude values, the column was attempted to be made as a column of floats and any that were left as objects were listed.


In [20]:
m = pd.to_numeric(df["Longitude"], errors="coerce").isna()

for i, val in df[m]["Longitude"].items():
    print(f"Row at index {i} has non-float Longitude value: {val}")

Row at index 76 has non-float Longitude value: (76..32373)
Row at index 382 has non-float Longitude value: (76. 62862)
Row at index 856 has non-float Longitude value: (77. 171533)


After listing the rows that couldn't be converted to a float, they were manually changed based on inference on what the value likely is supposed to be. For future implementations where a model may be trained online, sanitizing the coordinate values will likely be automated.


In [21]:
df.loc[76, "Longitude"] = 76.32373
df.loc[382, "Longitude"] = 76.62862
df.loc[856, "Longitude"] = 77.171533

Once the values were manually converted, the column is converted into a column of floats to confirm the type conversion.


In [22]:
df["Longitude"] = df["Longitude"].astype(float)
df["Longitude"].dtype

dtype('float64')

The units of longitude and latitude coordinates are listed to check for any necessary conversions.


In [23]:
df["Longitude Units"].value_counts()

Longitude Units
decimal degrees    2521
Name: count, dtype: int64

In [24]:
df["Latitude Units"].value_counts()

Latitude Units
decimal degrees    2521
Name: count, dtype: int64

All coordinates are measured in decimal degrees so there are no necessary conversions. For location-based computation and graphing purposes when using libraries like GeoPandas, the CRS should be EPSG:4326.


In [25]:
df.to_parquet(PROCESSED_DATA_DIR / "strandings.parquet", index=False)